# 3-10 다조건 기업 스크리너 — 시장·밸류에이션·재무 범위 + 산업/사업 키워드

**질문 예시:** "코스피에서 시총 1조 넘고 PBR 0.8 이하, 배당수익률 5% 넘는 종목 찾아줘",
"영업이익률 15% 넘고 부채비율 100% 이하인 회사 중 '원자력' 사업 하는 곳은?"

DeepSearch 자연어 쿼리는 `(조건) and (조건)` 형태로 여러 조건을 한 번에 걸 수 있습니다.
이 노트북은 **조건을 사전으로 적으면 쿼리를 만들어 실행**하고, 결과를 종목 정보·시세와 합쳐 표로 보여 줍니다.

| 조건 | 쿼리 형태 | 입력 단위 |
|---|---|---|
| 시장 | `('코스피' 상장 기업)`, 둘 다면 `(('코스피' 상장 기업) or ('코스닥' 상장 기업))` | KOSPI / KOSDAQ / ALL |
| 거래대금·시가총액 (기준일) | `(a <= 2026-11-24 시가총액 <= b)` | 억 원 |
| PER·PBR (기준일) | `(a <= 2026-11-24 PER <= b)` | 배 |
| 배당수익률 (기준일) | `(2026-11-24 배당수익률 >= 5)` | % (값 자체가 퍼센트) |
| 매출액·영업이익·당기순이익·자산총계 (사업연도) | `(2025 매출액 >= …)` | 억 원 |
| 영업이익률·당기순이익률 (사업연도) | `(2025 영업이익률 >= 0.15)` | % 입력 → 비율로 변환 |
| 부채비율 | 쿼리로는 오류 → 부채총계·자본총계를 받아 직접 계산 | % |
| 산업 키워드 | `('반도체' 산업 기업)` — 표준산업분류 이름 | 문자열 |
| 사업 키워드 | `SearchCompanyBusinessSummary('원자력', count=50)` — 정기보고서 사업 요약 문장 | 문자열 |

**확인한 제약 (2026-11-25):**
- 재무 조건은 **연결 기준**입니다(삼성전자·한화·CJ를 `GetFinancialStatements` 연결 값과 대조해 일치). `연결`/`개별`을 쿼리에 넣으면 0건이 되어 별도 기준은 고를 수 없습니다.
- `DPS/주가`로 배당수익률을 계산하는 옛 문법은 지금 종목명 검색으로 잘못 해석됩니다. `배당수익률`을 쓰세요.
- 키워드 조건(산업·사업)을 수치 조건과 한 쿼리에 묶으면 `TypeError`가 나는 경우가 있어, **수치 조건과 키워드를 따로 조회해 교집합**합니다.
- 사업 키워드 검색은 최대 50건(`count` 상한)까지만 봅니다. 흔한 단어는 결과가 잘립니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/3_price_screener/10_multi_condition_screener.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"   # 시장데이터(시총·PER 등) 기준일
FY = 2025             # 재무 조건의 사업연도

## 1. 조건 → 쿼리 만들기
범위는 `(최소, 최대)`로 적고, 한쪽만 줄 때는 다른 쪽을 `None`으로 둡니다.

In [4]:
MARKET_DATA = {"거래대금": "억", "시가총액": "억", "PER": "배", "PBR": "배", "배당수익률": "%그대로"}
FINANCIAL = {"매출액": "억", "영업이익": "억", "당기순이익": "억", "자산총계": "억",
             "영업이익률": "%비율", "당기순이익률": "%비율"}
MARKET_CLAUSE = {"KOSPI": "('코스피' 상장 기업)", "KOSDAQ": "('코스닥' 상장 기업)",
                 "ALL": "(('코스피' 상장 기업) or ('코스닥' 상장 기업))"}


def _conv(v, unit):
    if v is None:
        return None
    if unit == "억":
        return int(v * 100_000_000)
    if unit == "%비율":
        return round(v / 100, 6)
    return v


def _range(label, lo, hi):
    if lo is not None and hi is not None:
        return f"({lo} <= {label} <= {hi})"
    if lo is not None:
        return f"({label} >= {lo})"
    return f"({label} <= {hi})"


def build_query(cond):
    """조건 사전 → 수치 조건 쿼리 문자열. 키워드·부채비율은 run_screen 에서 따로 처리."""
    parts = [MARKET_CLAUSE[cond.get("market", "ALL")]]
    for key, unit in MARKET_DATA.items():
        if key in cond:
            lo, hi = (_conv(x, unit) for x in cond[key])
            parts.append(_range(f"{ASOF} {key}", lo, hi))
    for key, unit in FINANCIAL.items():
        if key in cond:
            lo, hi = (_conv(x, unit) for x in cond[key])
            parts.append(_range(f"{cond.get('fy', FY)} {key}", lo, hi))
    return " and ".join(parts)


build_query({"market": "KOSPI", "시가총액": (10000, None), "PBR": (0.1, 0.8), "배당수익률": (5, None)})

"('코스피' 상장 기업) and (2026-11-24 시가총액 >= 1000000000000) and (0.1 <= 2026-11-24 PBR <= 0.8) and (2026-11-24 배당수익률 >= 5)"

## 2. 실행 함수
1) 수치 조건 쿼리 → 2) 산업·사업 키워드 쿼리(따로) → 3) 부채비율(부채총계·자본총계로 계산) → 교집합 →
4) 일반법인·공기업 보통주만 남기고(ETF·우선주 제외) 기준일 시세를 붙입니다.

In [5]:
def value_cols(df):
    return [c for c in df.columns if c not in ("date", "symbol", "entity_name", "RowID") and not c.startswith("상장 기업")]


def run_screen(cond, show_query=True):
    steps = []
    q = build_query(cond)
    if show_query:
        print("수치 조건 쿼리:", q)
    base = ds_df(q)
    steps.append(("수치 조건", len(base)))
    if base.empty:
        print("→ 수치 조건에 맞는 종목이 없습니다(또는 조건 문법 오류).")
        return pd.DataFrame()
    keep = base[["symbol"] + value_cols(base)].drop_duplicates("symbol")

    if cond.get("industry_kw"):
        q2 = f"{MARKET_CLAUSE[cond.get('market', 'ALL')]} and ('{cond['industry_kw']}' 산업 기업)"
        ind = ds_df(q2)
        steps.append((f"산업 키워드 '{cond['industry_kw']}'", len(ind)))
        keep = keep[keep.symbol.isin(ind.symbol)]

    if cond.get("business_kw"):
        q3 = f"{MARKET_CLAUSE[cond.get('market', 'ALL')]} and SearchCompanyBusinessSummary('{cond['business_kw']}', count=50)"
        biz = ds_df(q3)
        steps.append((f"사업 키워드 '{cond['business_kw']}'", len(biz)))
        if len(biz) >= 50:
            print("  ⚠ 사업 키워드 결과가 상한(50)에 닿았습니다. 일부 회사가 빠졌을 수 있습니다.")
        keep = keep.merge(biz[["symbol", "summary_title"]].rename(columns={"summary_title": "사업 요약"}),
                          on="symbol", how="inner")

    if "부채비율" in cond:
        lo, hi = cond["부채비율"]
        fy = cond.get("fy", FY)
        debt = ds_df(f"{MARKET_CLAUSE[cond.get('market', 'ALL')]} and ({fy} 부채총계 >= 0) and ({fy} 자본총계 > 0)")
        debt["부채비율(%)"] = debt[f"부채총계 {fy}"] / debt[f"자본총계 {fy}"] * 100
        ok = debt[(debt["부채비율(%)"] >= (lo if lo is not None else -1e18)) &
                  (debt["부채비율(%)"] <= (hi if hi is not None else 1e18))]
        steps.append(("부채비율 (자본잠식 제외)", len(ok)))
        keep = keep.merge(ok[["symbol", "부채비율(%)"]], on="symbol", how="inner")

    snap = all_stocks(ASOF)[["symbol", "name", "market", "industry_name", "close", "market_cap", "value"]]
    out = keep.merge(snap, on="symbol", how="inner")   # 일반법인·공기업 보통주만 남는다
    steps.append(("교집합 → 보통주(ETF·우선주 제외)", len(out)))
    print(" → ".join(f"{a} {b}개" for a, b in steps))
    out = out.rename(columns={"name": "종목", "market": "시장", "industry_name": "업종", "close": "종가"})
    out["시가총액"] = out.pop("market_cap").map(krw)
    out["거래대금(당일)"] = out.pop("value").map(krw)
    out = out.drop(columns=[f"시가총액 {ASOF}", f"거래대금 {ASOF}"], errors="ignore")  # 시세 열과 중복
    for c in list(out.columns):           # 원 단위 금액 열은 읽기 쉽게
        if any(k in c for k in ("매출액", "영업이익 ", "당기순이익 ", "자산총계")):
            out[c] = out[c].map(krw)
        elif "이익률" in c:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
        elif any(k in c for k in ("PER", "PBR", "배당수익률", "부채비율")):
            out[c] = out[c].astype(float).round(2)
    front = ["종목", "시장", "업종", "종가", "시가총액"]
    return out[front + [c for c in out.columns if c not in front + ["symbol"]]].reset_index(drop=True)


print(PRICE_NOTICE)

※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). 거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).


## 3. 예시 ① 저PBR·고배당 대형주 (코스피)
시총 1조 원 이상, PBR 0.1~0.8배, 배당수익률 5% 이상, 흑자(PER > 0).

In [6]:
r1 = run_screen({"market": "KOSPI", "시가총액": (10000, None), "PBR": (0.1, 0.8),
                 "배당수익률": (5, None), "PER": (0.01, None)})
r1.sort_values(f"배당수익률 {ASOF}", ascending=False).reset_index(drop=True).head(20)

수치 조건 쿼리: ('코스피' 상장 기업) and (2026-11-24 시가총액 >= 1000000000000) and (2026-11-24 PER >= 0.01) and (0.1 <= 2026-11-24 PBR <= 0.8) and (2026-11-24 배당수익률 >= 5)


수치 조건 21개 → 교집합 → 보통주(ETF·우선주 제외) 18개


,종목,시장,업종,종가,시가총액,PER 2026-11-24,PBR 2026-11-24,배당수익률 2026-11-24,거래대금(당일)
0,에스엘,KOSPI,자동차용 기타 신품 부품 제조업,38200,1.8조,5.66,0.69,8.14,55억
1,기아,KOSPI,승용차 및 기타 여객용 자동차 제조업,95600,37.3조,4.93,0.61,7.87,"1,340억"
2,서울보증보험,KOSPI,보증 보험업,37000,2.6조,9.63,0.50,7.72,34억
3,한국금융지주,KOSPI,지주회사,127400,7.1조,3.68,0.62,7.15,182억
4,삼성카드,KOSPI,신용카드 및 할부금융업,40500,4.7조,6.69,0.49,6.94,54억
5,강원랜드,KOSPI,기타 사행시설 관리 및 운영업,14040,3.0조,8.93,0.71,6.85,78억
6,한일시멘트,KOSPI,시멘트 제조업,15140,1.1조,14.27,0.60,6.73,8억
7,신영증권,KOSPI,증권 중개업,121300,2.0조,6.52,0.50,6.56,7억
8,LX인터내셔널,KOSPI,상품 종합 중개업,31850,1.2조,8.08,0.41,6.43,35억
9,다우기술,KOSPI,포털 및 기타 인터넷 정보매개 서비스업,29850,1.3조,2.55,0.37,6.38,9억


## 4. 예시 ② 고마진·저부채 실적주 (코스피+코스닥)
당일 거래대금 10억 원 이상, 2025년 매출 1,000억 원 이상·영업이익률 15% 이상·순이익 흑자, 부채비율 100% 이하.

In [7]:
r2 = run_screen({"market": "ALL", "거래대금": (10, None), "매출액": (1000, None),
                 "영업이익률": (15, None), "당기순이익": (0.01, None), "부채비율": (None, 100)})
print(len(r2), "종목 (영업이익률 높은 순 20개)")
r2.assign(_m=r2[f"영업이익률 {FY}"].str.rstrip("%").astype(float)).sort_values("_m", ascending=False).drop(columns="_m").reset_index(drop=True).head(20)

수치 조건 쿼리: (('코스피' 상장 기업) or ('코스닥' 상장 기업)) and (2026-11-24 거래대금 >= 1000000000) and (2025 매출액 >= 100000000000) and (2025 당기순이익 >= 1000000) and (2025 영업이익률 >= 0.15)


수치 조건 87개 → 부채비율 (자본잠식 제외) 1334개 → 교집합 → 보통주(ETF·우선주 제외) 75개
75 종목 (영업이익률 높은 순 20개)


,종목,시장,업종,종가,시가총액,매출액 2025,당기순이익 2025,영업이익률 2025,부채비율(%),거래대금(당일)
0,맥쿼리인프라,KOSPI,신탁업 및 집합투자업,9600,4.6조,2.0조,2.0조,96.6%,9.48,71억
1,우리기술투자,KOSDAQ,금융리스업,4505,"3,595억","1,629억","1,105억",84.8%,21.19,24억
2,SK스퀘어,KOSPI,지주회사,715000,94.3조,10.5조,8.8조,84.1%,9.01,"4,165억"
3,클래시스,KOSDAQ,그 외 기타 의료용 기기 제조업,21600,1.4조,"3,368억","1,320억",50.6%,27.73,293억
4,알테오젠,KOSDAQ,의학 및 약학 연구개발업,159700,11.1조,"2,159억","1,452억",49.5%,53.76,"1,713억"
5,SK하이닉스,KOSPI,"다이오드, 트랜지스터 및 유사 반도체소자 제조업",1158000,818.5조,97.1조,42.9조,48.6%,45.95,3.9조
6,휴젤,KOSDAQ,생물학적 제제 제조업,202500,2.5조,"4,251억","1,470억",47.2%,9.86,87억
7,삼성바이오로직스,KOSPI,생물학적 제제 제조업,1290000,59.7조,4.6조,1.8조,45.4%,48.44,829억
8,한미반도체,KOSPI,반도체 제조용 기계 제조업,163600,15.6조,"5,767억","2,140억",43.6%,17.82,"1,024억"
9,파마리서치,KOSDAQ,의료용품 및 기타 의약 관련제품 제조업,264500,2.7조,"5,363억","1,683억",40.0%,44.21,374억


## 5. 예시 ③ 사업 키워드 '원자력' + 흑자 + PBR 1.5배 이하
사업 키워드는 회사가 정기보고서에 쓴 **사업 요약 문장**에서 찾습니다. 통계청 업종이 달라도(밸브·정비·건설 등)
원전 관련 사업을 적은 회사가 잡힙니다.

In [8]:
r3 = run_screen({"market": "ALL", "PBR": (None, 1.5), "영업이익": (0.01, None), "business_kw": "원자력"})
r3

수치 조건 쿼리: (('코스피' 상장 기업) or ('코스닥' 상장 기업)) and (2026-11-24 PBR <= 1.5) and (2025 영업이익 >= 1000000)


수치 조건 1188개 → 사업 키워드 '원자력' 20개 → 교집합 → 보통주(ETF·우선주 제외) 8개


,종목,시장,업종,종가,시가총액,PBR 2026-11-24,영업이익 2025,사업 요약,거래대금(당일)
0,금화피에스시,KOSDAQ,내부 전기배선 공사업,32750,"1,965억",0.50,438억,발전소 정비·플랜트 건설 중심의 인프라 시공 기업임,3억
1,HRS,KOSDAQ,합성수지 및 기타 플라스틱 물질 제조업,4500,736억,0.55,134억,실리콘 고무 컴파운드·가공제품 기반의 전문 제조사로 맞춤형 고부가 솔루션 제공,1억
2,한전KPS,KOSPI,일반전기 공사업,33550,1.5조,1.12,"1,401억","발전설비 O&M 공기업 기반, 원전·양수·신재생·해외 정비까지 확장",56억
3,한라IMS,KOSDAQ,선박 구성 부분품 제조업,12230,"2,093억",1.05,223억,선박 계측·제어 솔루션과 산업플랜트로 사업 확장 중,2억
4,일진파워,KOSDAQ,일반전기 공사업,8530,"1,286억",0.87,241억,"발전정비 기반, 원전·신성장·수소에너지로 다각화 추진 중임",6억
5,우진,KOSPI,산업처리공정 제어장비 제조업,8470,"1,710억",0.86,142억,산업용 계측·제어와 원자력 특화 솔루션을 기반으로 DX 확장,9억
6,수산인더스트리,KOSPI,일반전기 공사업,17650,"2,521억",0.45,431억,"원자력·화력·신재생 발전설비 종합정비 업체로 I&C·회전기·디지털 진단 역량 보유, 국내외 고객·계열",3억
7,콜마비앤에이치,KOSDAQ,기초 의약물질 및 생물학적 제제 제조업,7750,"2,230억",0.62,266억,"건기식·화장품 ODM 기반, 개별인정형 원료와 글로벌 공급망 확장",9억


## 6. 예시 ④ 산업 키워드 '반도체' + PER 0~20배 + 시총 3,000억 이상
산업 키워드는 **표준산업분류 이름**에 그 단어가 들어간 회사입니다(예: "다이오드, 트랜지스터 및 유사 반도체소자 제조업").
삼성전자처럼 업종명이 다른 회사("통신 및 방송 장비 제조업")는 빠지므로, 넓게 찾으려면 사업 키워드를 쓰세요.

In [9]:
r4 = run_screen({"market": "ALL", "PER": (0.01, 20), "시가총액": (3000, None), "industry_kw": "반도체"})
r4.sort_values(f"PER {ASOF}").reset_index(drop=True).head(20)

수치 조건 쿼리: (('코스피' 상장 기업) or ('코스닥' 상장 기업)) and (2026-11-24 시가총액 >= 300000000000) and (0.01 <= 2026-11-24 PER <= 20)


수치 조건 295개 → 산업 키워드 '반도체' 87개 → 교집합 → 보통주(ETF·우선주 제외) 7개


,종목,시장,업종,종가,시가총액,PER 2026-11-24,거래대금(당일)
0,프로텍,KOSDAQ,반도체 제조용 기계 제조업,67500,"6,075억",12.75,22억
1,코미코,KOSDAQ,반도체 제조용 기계 제조업,17040,"8,752억",16.91,87억
2,GST,KOSDAQ,반도체 제조용 기계 제조업,46250,"8,524억",18.26,88억
3,SK하이닉스,KOSPI,"다이오드, 트랜지스터 및 유사 반도체소자 제조업",1158000,818.5조,18.66,3.9조
4,HD현대에너지솔루션,KOSPI,기타 반도체소자 제조업,71700,"8,030억",19.27,64억
5,미코,KOSDAQ,기타 반도체소자 제조업,18350,"6,132억",19.58,62억
6,에스티아이,KOSDAQ,반도체 제조용 기계 제조업,20700,"3,210억",19.81,13억


## 결과 읽는 법과 한계
- **PER 조건은 음수(적자)도 포함합니다.** `PER <= 10`만 걸면 적자 기업이 섞이니 최소값(0.01)을 함께 주세요.
- **재무는 연결 기준, 사업연도 단위**입니다. 분기 실적 조건은 이 쿼리로 걸 수 없습니다.
- **결과는 현재 상장 종목 기준**이고 시세·거래대금은 KRX 기준입니다(NXT 미포함).
- 사업 키워드는 최대 50건만 보고, 요약 문장에 단어가 있느냐로 판단하므로 매출 비중은 따로 확인해야 합니다(→ 1-05 사업 프로파일).